In [24]:
import pandas as pd
import numpy as np
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import textstat

df = pd.read_csv('../output/lada/lsat_qa_lada_3k.csv')

In [25]:
from collections import defaultdict

def cap_by_type(df_in, type_col='type', max_per_type=25):
    # preserve the original row order while limiting each type to max_per_type
    counts = defaultdict(int)
    keep_idx = []
    for idx, t in zip(df_in.index, df_in[type_col]):
        if counts[t] < max_per_type:
            keep_idx.append(idx)
            counts[t] += 1
    return df_in.loc[keep_idx]

# F1 analysis

In [26]:
top_f1 = df.sort_values('Factor_1', ascending=False).head(100)

top_f1 = cap_by_type(top_f1)

print(top_f1['type'].value_counts())

top_f2 = df.sort_values('Factor_2', ascending=False).head(100)

top_f2 = cap_by_type(top_f2)

print(top_f2['type'].value_counts())

top_f3 = df.sort_values('Factor_3', ascending=False).head(100)

top_f3 = cap_by_type(top_f3)

print(top_f3['type'].value_counts())

type
ar    25
Name: count, dtype: int64
type
ar    25
Name: count, dtype: int64
type
ar    25
Name: count, dtype: int64


In [27]:
word_pattern = re.compile(r"\b\w+\b")
sentiment_analyzer = SentimentIntensityAnalyzer()

def compute_text_metrics(series, include_sentiment_label=False):
    metrics = []
    for text in series.fillna(""):
        text_str = str(text)
        tokens = word_pattern.findall(text_str.lower())
        token_lengths = [len(token) for token in tokens]
        question_length = len(tokens)
        avg_length = float(np.mean(token_lengths)) if token_lengths else 0.0
        burstiness = float(np.std(token_lengths) / avg_length) if token_lengths and avg_length else 0.0
        if token_lengths:
            counts = Counter(tokens)
            total = sum(counts.values())
            probs = np.array(list(counts.values()), dtype=float) / total
            entropy = float(-np.sum(probs * np.log(probs)))
            perplexity = float(np.exp(entropy))
        else:
            perplexity = 0.0
        flesch_kincaid = float(textstat.flesch_kincaid_grade(text_str)) if text_str.strip() else 0.0
        sentiment_scores = sentiment_analyzer.polarity_scores(text_str) if text_str.strip() else {"compound": 0.0}
        compound_sentiment = float(sentiment_scores.get("compound", 0.0))
        record = {
            "question": text_str,
            "question_length": question_length,
            "average_word_length": avg_length,
            "burstiness": burstiness,
            "perplexity": perplexity,
            "flesch_kincaid_grade": flesch_kincaid,
            "sentiment_compound": compound_sentiment
        }
        if include_sentiment_label:
            if compound_sentiment >= 0.05:
                sentiment_label = "positive"
            elif compound_sentiment <= -0.05:
                sentiment_label = "negative"
            else:
                sentiment_label = "neutral"
            record["sentiment_label"] = sentiment_label
        metrics.append(record)
    return pd.DataFrame(metrics)

In [28]:
df_metrics = compute_text_metrics(df['question'])
top_f1_metrics = compute_text_metrics(top_f1['question'])
top_f2_metrics = compute_text_metrics(top_f2['question'])
top_f3_metrics = compute_text_metrics(top_f3['question'])


summary_columns = ["question_length", "average_word_length", "burstiness", "perplexity", "flesch_kincaid_grade", "sentiment_compound"]
summary_df = pd.concat(
    [
        df_metrics[summary_columns].mean().rename("overall_mean"),
        top_f1_metrics[summary_columns].mean().rename("top_f1_mean"),
        top_f2_metrics[summary_columns].mean().rename("top_f2_mean"),
        top_f3_metrics[summary_columns].mean().rename("top_f3_mean")
    ],
    axis=1
)

display(summary_df)

for label, metrics_df in [
    ("overall", df_metrics),
    ("top_f1", top_f1_metrics),
    ("top_f2", top_f2_metrics),
    ("top_f3", top_f3_metrics)
]:
    print(f"\nSample metrics for {label} questions:")
    display(metrics_df.head(5))
    if "sentiment_label" in metrics_df.columns:
        print("Sentiment label distribution:")
        display(metrics_df['sentiment_label'].value_counts(normalize=True).rename(lambda x: f"{x} ({label})"))

,overall_mean,top_f1_mean,top_f2_mean,top_f3_mean
question_length,123.198238,118.000000,123.360000,120.720000
average_word_length,4.549748,4.572350,4.591700,4.599620
burstiness,0.555709,0.556920,0.548843,0.549219
perplexity,42.742515,42.615926,42.668256,42.413230
flesch_kincaid_grade,9.030084,9.320334,9.085766,9.109305
sentiment_compound,0.212732,0.237016,0.335628,0.418680



Sample metrics for overall questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,A bakery makes exactly three kinds of cookie—o...,124,4.241935,0.445452,41.698082,7.092125,-0.2960
1,A bakery makes exactly three kinds of cookie—o...,119,4.252101,0.468592,40.049018,6.884286,-0.1531
2,A bakery makes exactly three kinds of cookie—o...,132,4.174242,0.461091,40.908061,7.330476,-0.2960
3,A bakery makes exactly three kinds of cookie—o...,122,4.303279,0.452103,39.554095,7.242338,0.1531
4,A bakery makes exactly three kinds of cookie—o...,121,4.264463,0.457980,40.225494,7.222381,0.2263



Sample metrics for top_f1 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,"A company's six vehicles—a hatchback, a limous...",140,4.435714,0.554981,37.571949,11.016111,0.1406
1,A television programming director is schedulin...,123,4.813008,0.511236,50.906860,8.754107,-0.7003
2,"Of the eight students—George, Helen, Irving, K...",123,4.650407,0.491262,54.713565,12.799248,0.4215
3,"Four employees—Jackson, Larabee, Paulson, and ...",125,4.408000,0.591646,52.306108,11.974426,-0.3818
4,"Seven workers—Quinn, Ruiz, Smith, Taylor, Verm...",101,4.613861,0.527356,38.137233,9.200556,0.8805



Sample metrics for top_f2 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,A record producer is planning the contents of ...,99,5.313131,0.548010,37.417483,12.884612,0.8402
1,"An editor will edit seven articles, one at a t...",108,4.222222,0.602396,49.347183,9.152363,0.3182
2,"During a recital, two pianists—Wayne and Zara—...",125,4.904000,0.544983,46.053795,9.022927,-0.2960
3,"Five students—Manolo, Nadia, Owen, Peng, and R...",102,4.647059,0.530237,38.270535,8.295242,0.9556
4,A concert promoter is filling the six slots at...,119,4.579832,0.494760,44.872689,5.923333,0.6597



Sample metrics for top_f3 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,A record producer is planning the contents of ...,99,5.333333,0.547463,38.277362,13.005020,0.8402
1,"Four art historians—Farley, Garcia, Holden, an...",119,4.873950,0.565843,39.492530,10.970357,0.6369
2,"Exactly five students—Grecia, Hakeem, Joe, Kat...",149,4.557047,0.508081,58.689127,8.027126,0.5106
3,A professor must determine the order in which ...,82,5.304878,0.480768,42.055249,11.735000,0.5095
4,"During a single week, from Monday through Frid...",137,4.656934,0.559739,41.712890,8.638111,0.1531
